# Equity Intelligence Research Platform — Quickstart

Runs in any Python 3.9+ environment: local machine, Jupyter Lab, VS Code,
Google Colab, Kaggle, AWS SageMaker, Azure ML, or any Linux/Windows/macOS shell.

**What this does:**
1. Installs dependencies
2. Configures your LLM API key (one is enough — Groq is free)
3. Runs the full 17-agent pipeline for any public company
4. Saves an institutional DOCX research report to disk

**Time:** 3–8 min per company. **Minimum:** Python 3.9, one LLM key.

## 1 — Install Dependencies

In [ ]:
# Works on pip-based environments (local, Colab, Kaggle, SageMaker, Azure ML, etc.)
# Re-run this cell any time you create a fresh environment.

import subprocess, sys

packages = [
    "openai>=1.40.0", "anthropic>=0.34.0", "groq>=0.11.0",
    "google-generativeai>=0.7.0",
    "yfinance>=0.2.40", "pandas>=2.1.0", "numpy>=1.26.0", "polars>=0.20.0",
    "httpx>=0.27.0", "requests>=2.31.0", "beautifulsoup4>=4.12.0", "lxml>=5.0.0",
    "PyMuPDF>=1.24.0", "pdfplumber>=0.11.0",
    "pydantic>=2.7.0", "pydantic-settings>=2.3.0", "python-dotenv>=1.0.0",
    "duckdb>=1.0.0", "pyarrow>=16.0.0",
    "python-docx>=1.1.0", "plotly>=5.22.0", "matplotlib>=3.8.0", "openpyxl>=3.1.0",
    "loguru>=0.7.0", "pyyaml>=6.0.1", "rich>=13.7.0",
    "langgraph>=0.2.0", "langchain-core>=0.2.0",
    "scipy>=1.13.0", "statsmodels>=0.14.0",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("Installation complete.")

In [ ]:
import importlib
checks = {
    "yfinance": "yfinance",
    "pydantic": "pydantic",
    "loguru": "loguru",
    "python-docx": "docx",
    "groq": "groq",
    "openai": "openai",
}
all_ok = True
for label, mod in checks.items():
    try:
        importlib.import_module(mod)
        print(f"  ok  {label}")
    except ImportError:
        print(f"  MISSING  {label}")
        all_ok = False

print("\nAll packages ready." if all_ok else "\nRe-run the install cell.")

## 2 — Make the Platform Importable

The `equity_research/` directory must be on `sys.path`.

**If you opened this notebook from inside the `equity_research/` folder** (i.e. `equity_research/quickstart.ipynb`),
the parent directory is added automatically below.

**If you placed the notebook elsewhere**, update `PLATFORM_ROOT` to the folder
that *contains* `equity_research/`.

In [ ]:
import sys
from pathlib import Path

# The notebook lives inside equity_research/, so its parent is the platform root.
# Override PLATFORM_ROOT if you moved the notebook somewhere else.
PLATFORM_ROOT = Path("__file__" if "__file__" in dir() else ".").resolve().parent

# Fallback: walk upward until we find equity_research/__init__.py
candidate = Path(".").resolve()
for _ in range(5):
    if (candidate / "equity_research" / "__init__.py").exists():
        PLATFORM_ROOT = candidate
        break
    if (candidate / "__init__.py").exists() and candidate.name == "equity_research":
        PLATFORM_ROOT = candidate.parent
        break
    candidate = candidate.parent

if str(PLATFORM_ROOT) not in sys.path:
    sys.path.insert(0, str(PLATFORM_ROOT))

if (PLATFORM_ROOT / "equity_research" / "__init__.py").exists():
    print(f"Platform root: {PLATFORM_ROOT}")
    print(f"equity_research/ found — ready to import.")
else:
    print("ERROR: equity_research/__init__.py not found.")
    print(f"Searched: {PLATFORM_ROOT}")
    print("Set PLATFORM_ROOT manually to the folder containing equity_research/.")

## 3 — Configure API Keys

The platform auto-detects available providers in this order:
`Groq → OpenAI → Anthropic → Gemini → Together → OpenRouter → Ollama → Template`

**One key is enough.** Groq has a free tier — get it at [console.groq.com](https://console.groq.com).

**Alternative:** create a `.env` file next to this notebook instead of hardcoding keys here:
```
GROQ_API_KEY=gsk_...
OPENAI_API_KEY=sk-...
```

In [ ]:
import os
from pathlib import Path

# ── Option A: hardcode keys here (clear before sharing the notebook) ──
API_KEYS = {
    "GROQ_API_KEY":       "",   # Free: https://console.groq.com/keys
    "OPENAI_API_KEY":     "",
    "ANTHROPIC_API_KEY":  "",
    "GOOGLE_API_KEY":     "",
    "TOGETHER_API_KEY":   "",
    "OPENROUTER_API_KEY": "",
}

# ── Option B: load from .env file (recommended for local use) ─────────
try:
    from dotenv import load_dotenv
    # Looks for .env in the platform root and current directory
    load_dotenv(PLATFORM_ROOT / ".env", override=False)
    load_dotenv(Path(".") / ".env", override=False)
except ImportError:
    pass

# Apply any keys set above (non-empty values override .env)
for var, val in API_KEYS.items():
    if val.strip():
        os.environ[var] = val.strip()

# Show which providers have keys
for var in API_KEYS:
    status = "set" if os.environ.get(var) else "not set"
    print(f"  {var}: {status}")

In [ ]:
from equity_research.core.llm_manager import LLMManager

llm = LLMManager()
info = llm.get_backend_info()
print(f"Active LLM backend: {info}")

if "template" in info.lower():
    print("\nNo LLM API key detected — running in template mode.")
    print("The pipeline will complete but narrative sections will use placeholder text.")
    print("Add a key in the cell above for full LLM-powered output.")

## 4 — Set Output Directory

Reports, raw filings, and audit logs are written here.
Defaults to `~/equity_research_reports/` on any OS.

In [ ]:
import os
from pathlib import Path

# Change this to any writable path on your system
OUTPUT_DIR = Path.home() / "equity_research_reports"

# Uncomment to override:
# OUTPUT_DIR = Path("/tmp/reports")          # Linux/macOS tmp
# OUTPUT_DIR = Path("C:/Reports")            # Windows absolute
# OUTPUT_DIR = Path("./reports")             # Relative to notebook
# OUTPUT_DIR = Path("/content/reports")      # Google Colab
# OUTPUT_DIR = Path("/kaggle/working")       # Kaggle

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["ER_REPORTS_DIR"] = str(OUTPUT_DIR)
print(f"Reports will be saved to: {OUTPUT_DIR}")

## 5 — Run Research

Set the company name and ticker, then run. The full 17-agent pipeline runs end-to-end.

**Examples**

| Company | `COMPANY_NAME` | `TICKER` |
|---------|---------------|----------|
| Apple | `Apple Inc` | `AAPL` |
| Microsoft | `Microsoft` | `MSFT` |
| TCS (NSE) | `Tata Consultancy Services` | `TCS.NS` |
| Infosys (NSE) | `Infosys` | `INFY.NS` |
| Reliance (NSE) | `Reliance Industries` | `RELIANCE.NS` |
| HDFC Bank (NSE) | `HDFC Bank` | `HDFCBANK.NS` |
| Tesla | `Tesla` | `TSLA` |
| Samsung | `Samsung Electronics` | `005930.KS` |
| Nestlé | `Nestle` | `NESN.SW` |

In [ ]:
from equity_research.orchestrator.workflow import ResearchOrchestrator

COMPANY_NAME = "Apple Inc"
TICKER       = "AAPL"       # Leave blank to let the platform auto-resolve

print(f"Starting: {COMPANY_NAME} ({TICKER or 'auto-resolve'})")
print("="*60)

orchestrator = ResearchOrchestrator()
result = orchestrator.research(
    company_name=COMPANY_NAME,
    ticker=TICKER,
    output_dir=OUTPUT_DIR,
)

print("="*60)
if result.get("error"):
    print(f"ERROR: {result['error']}")
else:
    print("Research complete.")

## 6 — View Results

In [ ]:
if result.get("error"):
    print(f"Pipeline error: {result['error']}")
else:
    ticker       = result.get("ticker", "N/A")
    company      = result.get("company_name", "N/A")
    rating       = result.get("investment_rating", "N/A")
    target       = result.get("target_price")
    current      = result.get("current_price")
    upside       = result.get("upside_pct")
    risk         = result.get("overall_risk_score", 0)
    duration     = result.get("duration_seconds", 0)
    agents_done  = result.get("agents_completed", 0)
    agents_fail  = result.get("agents_failed", 0)
    findings     = result.get("total_findings", 0)
    critical     = result.get("critical_findings", 0)
    val_passed   = result.get("validation_passed", "N/A")
    report       = result.get("report_path", "N/A")

    print(f"{'='*55}")
    print(f"  {company} ({ticker})")
    print(f"{'='*55}")
    print(f"  Rating          : {rating}")
    if target:
        print(f"  Target Price    : ${target:.2f}")
    if current:
        print(f"  Current Price   : ${current:.2f}")
    if upside is not None:
        print(f"  Upside          : {upside:+.1f}%")
    print(f"  Risk Score      : {risk:.0f} / 100")
    print(f"  Duration        : {duration:.0f}s")
    print(f"  Agents          : {agents_done} completed, {agents_fail} failed")
    print(f"  Findings        : {findings} total, {critical} critical")
    print(f"  Validation      : {'PASS' if val_passed else 'FAIL'}")
    print(f"  Report          : {report}")
    print(f"{'='*55}")

    print("\nAgent risk scores:")
    for agent_id, score in sorted(result.get("agent_risk_scores", {}).items()):
        bar = "#" * int(score / 5)
        flag = "  <-- high" if score > 65 else ""
        print(f"  {agent_id:<35} {score:5.0f}/100  {bar}{flag}")

## 7 — Open or Download the Report

The DOCX is already saved to disk. The cell below opens it on desktop environments
and offers a download link on Colab/Kaggle.

In [ ]:
import os, sys
from pathlib import Path

report_path = result.get("report_path")

if not report_path or not Path(report_path).exists():
    # Scan output dir for any .docx as fallback
    docx_files = sorted(OUTPUT_DIR.rglob("*.docx"), key=lambda p: p.stat().st_mtime, reverse=True)
    report_path = str(docx_files[0]) if docx_files else None

if report_path and Path(report_path).exists():
    print(f"Report: {report_path}")
    size_kb = Path(report_path).stat().st_size / 1024
    print(f"Size  : {size_kb:.0f} KB")

    # Desktop environments — open with the system default application
    if sys.platform == "win32":
        os.startfile(report_path)
    elif sys.platform == "darwin":
        os.system(f'open "{report_path}"')
    elif sys.platform.startswith("linux") and "DISPLAY" in os.environ:
        os.system(f'xdg-open "{report_path}" &')
    else:
        print("(No display detected — file saved at path above.)")

    # Colab / Kaggle — offer browser download
    try:
        from google.colab import files
        print("Triggering browser download...")
        files.download(report_path)
    except ImportError:
        pass  # Not Colab — desktop open already handled above

else:
    print("Report not found. Check the log output from Step 5 for errors.")

## Batch Mode — Multiple Companies

In [ ]:
# Each company gets its own timestamped output folder.
# Reports accumulate in OUTPUT_DIR across runs.

COMPANIES = [
    {"name": "Apple Inc",                   "ticker": "AAPL"},
    {"name": "Tata Consultancy Services",   "ticker": "TCS.NS"},
    # {"name": "Microsoft",                 "ticker": "MSFT"},
    # {"name": "Infosys",                   "ticker": "INFY.NS"},
]

from equity_research.orchestrator.workflow import ResearchOrchestrator

orch = ResearchOrchestrator()
batch_results = []

for co in COMPANIES:
    print(f"\nResearching: {co['name']} ({co['ticker']})")
    r = orch.research(company_name=co["name"], ticker=co["ticker"], output_dir=OUTPUT_DIR)
    batch_results.append(r)
    rating = r.get("investment_rating", "N/A")
    upside = r.get("upside_pct") or 0
    risk   = r.get("overall_risk_score", 0)
    err    = r.get("error", "")
    status = f"{rating} | upside {upside:+.1f}% | risk {risk:.0f}" if not err else f"ERROR: {err}"
    print(f"  -> {status}")

print(f"\nBatch complete. {len(batch_results)} companies researched.")
print(f"All reports in: {OUTPUT_DIR}")

In [ ]:
# List all generated reports
reports = sorted(OUTPUT_DIR.rglob("*.docx"), key=lambda p: p.stat().st_mtime, reverse=True)
print(f"{len(reports)} report(s) in {OUTPUT_DIR}:\n")
for r in reports:
    size_kb = r.stat().st_size / 1024
    print(f"  {r.name}  ({size_kb:.0f} KB)")

In [ ]:
# Pack all DOCX reports into a ZIP (useful on headless servers / Colab / Kaggle)
import zipfile
from datetime import datetime

zip_path = OUTPUT_DIR / f"reports_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for rp in OUTPUT_DIR.rglob("*.docx"):
        zf.write(rp, rp.name)

print(f"ZIP created: {zip_path}  ({zip_path.stat().st_size / 1024:.0f} KB)")

# Trigger browser download on Colab / Kaggle
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    pass  # On desktop, the file is at zip_path

## Configuration Reference

Defaults live in `equity_research/config.yaml`. Override any value via environment variable
using the `ER_` prefix and `__` for nesting:

```python
import os
os.environ["ER_MODELING__FORECAST_YEARS"]   = "3"    # default 5
os.environ["ER_LLM__MAX_TOKENS"]            = "2048"  # default 4096
os.environ["ER_LLM__TEMPERATURE"]           = "0.0"
os.environ["ER_ACQUISITION__YEARS_HISTORY"] = "5"    # default 7
os.environ["ER_REPORT__FIRM_NAME"]          = "My Research Firm"
```

Or edit `config.yaml` directly before importing the platform.

### Ticker Formats by Exchange

| Exchange | Format | Example |
|----------|--------|---------|
| NYSE / NASDAQ | plain | `AAPL`, `MSFT` |
| NSE (India) | `.NS` suffix | `TCS.NS`, `INFY.NS` |
| BSE (India) | `.BO` suffix | `500325.BO` |
| LSE (UK) | `.L` suffix | `AZN.L` |
| TSX (Canada) | `.TO` suffix | `RY.TO` |
| ASX (Australia) | `.AX` suffix | `CBA.AX` |
| HKEX | `.HK` suffix | `0700.HK` |
| TSE (Japan) | `.T` suffix | `7203.T` |
| KRX (Korea) | `.KS` suffix | `005930.KS` |
| SIX (Switzerland) | `.SW` suffix | `NESN.SW` |

### LLM Provider Keys

| Provider | Env Var | Free Tier | Speed |
|----------|---------|-----------|-------|
| **Groq** | `GROQ_API_KEY` | Yes — [console.groq.com](https://console.groq.com) | Fastest |
| OpenAI | `OPENAI_API_KEY` | No | Fast |
| Anthropic | `ANTHROPIC_API_KEY` | No | Fast |
| Google Gemini | `GOOGLE_API_KEY` | Yes (limited) | Fast |
| Together AI | `TOGETHER_API_KEY` | Yes (limited) | Fast |
| OpenRouter | `OPENROUTER_API_KEY` | Yes (pay-per-token) | Varies |
| Ollama (local) | none | Yes — [ollama.ai](https://ollama.ai) | Depends on hardware |

### Troubleshooting

| Problem | Fix |
|---------|-----|
| `ModuleNotFoundError: equity_research` | Re-run Step 2 path setup |
| Template mode / no LLM | Add an API key in Step 3 |
| `yfinance` returns empty data | Check ticker suffix (`.NS`, `.L`, etc.) |
| Rate limit / 429 errors | Platform auto-retries. If persistent, switch provider. |
| `Report not found` | Check Step 5 log for agent failures. Missing key data can abort report. |
| Slow on local Ollama | Reduce `ER_LLM__MAX_TOKENS` or use a smaller model (`qwen2.5:7b`). |